# MossFormer

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from utils.clear_memory import clear_memory

# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
input_dir_Pitt = Path('../ad_detection/data/raw/Pitt-origin')
output_dir_Pitt = Path('../ad_detection/data/denoised/Pitt-origin-MossFormer')

control_files_Pitt = list((input_dir_Pitt / 'Control').glob('*.wav')) + list((input_dir_Pitt / 'Control').glob('*.mp3'))
dementia_files_Pitt = list((input_dir_Pitt / 'Dementia').glob('*.wav')) + list((input_dir_Pitt / 'Dementia').glob('*.mp3'))

input_dir_Lu = Path('../ad_detection/data/raw/Lu')
output_dir_Lu = Path('../ad_detection/data/denoised/Lu-MossFormer')

control_files_Lu = list((input_dir_Lu / 'Control').glob('*.wav')) + list((input_dir_Lu / 'Control').glob('*.mp3'))
dementia_files_Lu = list((input_dir_Lu / 'Dementia').glob('*.wav')) + list((input_dir_Lu / 'Dementia').glob('*.mp3'))

## Load MossFormer Model

In [ ]:
from clearvoice import ClearVoice

# Use MossFormer model
model_name = 'MossFormerGAN_SE_16K'
target_sr = 16000  # Target sample rate

myClearVoice = ClearVoice(
    task='speech_enhancement',
    model_names=[model_name]
)

## Denoise Function

In [ ]:
def denoise_audio(audio_path, model, target_sr=16000):
    """
    Apply MossFormer for speech denoising and enhancement

    Args:
        audio_path: Input audio file path
        model: ClearVoice model instance
        target_sr: Target sample rate (16000)

    Returns:
        denoised_audio: Denoised audio numpy array
        sr: Sample rate
    """
    # Load audio
    audio, sr = sf.read(str(audio_path))

    # If multi-channel, convert to mono first (before resampling)
    if len(audio.shape) == 2:
        audio = np.mean(audio, axis=1)

    # Resample to target sample rate (using scipy, more stable)
    if sr != target_sr:
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)

    # Ensure float32 type
    audio = audio.astype(np.float32)

    # Convert to [batch, length] format
    audio = np.reshape(audio, [1, audio.shape[0]])

    # Apply MossFormer denoising
    with torch.no_grad():
        output_wav = model(audio, online_write=False)

    # output_wav shape: [batch, length]
    return output_wav[0, :], target_sr

In [ ]:
def batch_denoise(files, output_subdir, model, target_sr, group_name):
    """
    Batch denoising (clears memory before and after each file)

    Args:
        files: List of audio files to process
        output_subdir: Output subdirectory
        model: ClearVoice model instance
        target_sr: Target sample rate
        group_name: Group name (for progress display)
    """
    # Create output directory
    output_subdir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    skip_count = 0
    fail_count = 0

    for audio_file in tqdm(files, desc=f"Denoising {group_name}"):
        # Unify output as .wav format
        output_file = output_subdir / (audio_file.stem + '.wav')

        # Skip already processed files
        if output_file.exists():
            skip_count += 1
            continue

        try:
            clear_memory()

            # Denoise
            denoised_audio, sr = denoise_audio(audio_file, model, target_sr)

            # Save (16-bit integer format)
            sf.write(str(output_file), denoised_audio, sr, subtype='PCM_16')
            success_count += 1

            del denoised_audio
            clear_memory()

        except Exception as e:
            fail_count += 1
            print(f"\n✗ Failed: {audio_file.name}: {e}")
            clear_memory()

    # Print statistics
    print(f"\n{group_name} processing complete:")
    print(f"Success: {success_count}")
    print(f"Skipped: {skip_count}")
    print(f"Failed: {fail_count}")
    print(f"Total: {len(files)}")

## Pitt Denoise

In [ ]:
batch_denoise(
    dementia_files_Pitt,
    output_dir_Pitt / 'Dementia',
    myClearVoice,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files_Pitt,
    output_dir_Pitt / 'Control',
    myClearVoice,
    target_sr=target_sr,
    group_name='Control'
)

## Lu Denoise

In [ ]:
batch_denoise(
    dementia_files_Lu,
    output_dir_Lu / 'Dementia',
    myClearVoice,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files_Lu,
    output_dir_Lu / 'Control',
    myClearVoice,
    target_sr=target_sr,
    group_name='Control'
)